In [74]:
using Pkg
Pkg.activate("./insight_calc")
using D4M, DataFrames, DotEnv
include("insight_calc/src/automata.jl")
include("insight_calc/src/storage.jl")

  Activating project at `~/populi.Wk/InsightCircle/insight_calc`
[ Info: ENV_PATH==/home/gcr/populi.Wk/InsightCircle/.env


_rowsToAssocEAV (generic function with 1 method)

In [75]:
Pkg.status()

Project InsightCalc v0.1.0
Status `~/populi.Wk/InsightCircle/insight_calc/Project.toml`
  [67c07d97] Automa v1.1.0
  [ca196bdc] D4M v0.6.11 `../../D4M.jl`
  [a93c6f00] DataFrames v1.8.1
  [4dc1fcf4] DotEnv v1.0.0
  [55e21f81] GoogleCloud v0.11.0
  [cd3eb016] HTTP v1.11.0
  [0f8b85d8] JSON3 v1.14.3
⌃ [df9a0d86] Oxygen v1.7.5
  [856f2bd8] StructTypes v1.11.0
  [ade2ca70] Dates v1.11.0
  [56ddb016] Logging v1.11.0
  [44cfe95a] Pkg v1.11.0
  [8dfed614] Test v1.11.0
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [76]:
session = getServer()
fieldnames(typeof(session))

(:project, :dataset, :session)

In [77]:
listTables()

[ Info: Session is unauthorized. Fetching new token...


1-element Vector{String}:
 "yt_metadata"

In [ ]:
srv     = getServer()
jobId   = _submitJob(srv.session, srv.project, """
    SELECT id, COUNT(*) AS n
    FROM `creator-d4m-2026-1774038056.insight_metadata.yt_metadata`
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY n DESC
""")
url     = "$_BQ_BASE/projects/$(srv.project)/queries/$jobId?maxResults=100&location=us-central1"
payload = JSON3.read(HTTP.get(url, _authHeaders(srv.session)).body)

isempty(get(payload, :rows, [])) ? "No duplicates" : begin
    cols = [String(f.name) for f in payload.schema.fields]
    rows = [[isnothing(c.v) ? "" : String(c.v) for c in row.f] for row in payload.rows]
    DataFrame(permutedims(hcat(rows...)), cols)
end


LoadError: MethodError: no method matching String(::JSON3.Array{JSON3.Object, Vector{UInt8}, SubArray{UInt64, 1, Vector{UInt64}, Tuple{UnitRange{Int64}}, true}})
The type `String` exists, but no method is defined for this combination of argument types when trying to construct it.

[0mClosest candidates are:
[0m  String([91m::Vector{UInt8}[39m)
[0m[90m   @[39m [90mBase[39m [90mstrings/[39m[90m[4mstring.jl:72[24m[39m
[0m  String([91m::OpenSSL.BigNum[39m)
[0m[90m   @[39m [32mOpenSSL[39m [90m~/.julia/packages/OpenSSL/2SUGA/src/[39m[90m[4mOpenSSL.jl:2906[24m[39m
[0m  String([91m::Symbol[39m)
[0m[90m   @[39m [90mBase[39m [90mstrings/[39m[90m[4mstring.jl:117[24m[39m
[0m  ...
